# Component Presence Detector with Synthetic Negatives

This notebook trains a binary classifier to detect PCL presence/absence for each connector.

**Key Assumptions:**
- Images in `Data/connectors/connX/` are PNG or JPG (grayscale or RGB)
- All crops for a given connector have the same size
- Train/val split: 90/10 with fixed seed
- Model: ResNet18 pretrained, adapted for grayscale input
- Threshold calibration: 99.5th percentile on real OK validation images


## 1. Setup & Imports

**Note:** This notebook expects images to be in Google Drive at `/content/drive/MyDrive/Project Work/Data/connectors/`


In [1]:
# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully")
except ImportError:
    print("Not running in Colab, skipping Drive mount")

import os
import json
import random
import warnings
from pathlib import Path
from typing import Tuple, List, Dict, Optional, Union
from collections import defaultdict

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import resnet18
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm import tqdm

# Set seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True

# Paths
BASE_DIR = Path(".")
MASK_CONFIG_PATH = BASE_DIR / "mask_config.json"
CONNECTORS_ROOT = Path("/Data/connectors")
OUTPUT_ROOT = BASE_DIR / "outputs" / "pcl_presence"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Default hyperparameters
DEFAULT_INPAINT_RADIUS = 3
DEFAULT_MASK_DILATE_PX = 3
DEFAULT_TRAIN_VAL_SPLIT = 0.9
DEFAULT_THRESHOLD_PERCENTILE = 99.5
DEFAULT_EPOCHS = 15
DEFAULT_BATCH_SIZE = 32
DEFAULT_LEARNING_RATE = 1e-4


Not running in Colab, skipping Drive mount
Using device: cuda


## 2. Load mask_config.json and Build Masks


In [ ]:
def load_mask_config(config_path: Path) -> Dict:
    """Load mask configuration JSON."""
    with open(config_path, 'r') as f:
        return json.load(f)

def get_connector_directories(connectors_root: Path) -> List[str]:
    """Get list of connector directories."""
    connectors = []
    for item in connectors_root.iterdir():
        if item.is_dir() and item.name.startswith('conn'):
            connectors.append(item.name)
    return sorted(connectors)

def create_mask_from_polygon(polygon_points: List[Dict], image_width: int, image_height: int) -> np.ndarray:
    """Create binary mask from polygon points (relative coordinates)."""
    mask = np.zeros((image_height, image_width), dtype=np.uint8)
    
    # Convert relative coordinates to absolute
    points = []
    for pt in polygon_points:
        x = int(pt['x_rel'] * image_width)
        y = int(pt['y_rel'] * image_height)
        points.append([x, y])
    
    points = np.array(points, dtype=np.int32)
    cv2.fillPoly(mask, [points], 255)
    return mask

def create_mask_from_bbox(bbox: List[int], image_width: int, image_height: int) -> np.ndarray:
    """Create binary mask from bounding box [x1, y1, x2, y2]."""
    mask = np.zeros((image_height, image_width), dtype=np.uint8)
    x1, y1, x2, y2 = bbox
    mask[y1:y2, x1:x2] = 255
    return mask

def build_connector_masks(config: Dict, connectors: List[str], connectors_root: Path) -> Dict[str, Dict]:
    """Build binary masks for each connector and verify image sizes."""
    connector_info = {}
    
    for conn_id in connectors:
        if conn_id not in config:
            warnings.warn(f"No mask config found for {conn_id}, skipping.")
            continue
        
        conn_config = config[conn_id]
        
        # Get image size from config (required)
        if 'image_size' not in conn_config:
            warnings.warn(f"No image_size in config for {conn_id}, skipping.")
            continue
        
        expected_width = conn_config['image_size']['width']
        expected_height = conn_config['image_size']['height']
        
        # Verify images exist and check sizes
        conn_dir = connectors_root / conn_id
        img_files = list(conn_dir.glob("*.png")) + list(conn_dir.glob("*.jpg"))
        if not img_files:
            warnings.warn(f"No images found for {conn_id}, skipping.")
            continue
        
        # Check first few images to verify size matches
        size_mismatch = False
        for img_file in img_files[:5]:  # Check first 5 images
            img = cv2.imread(str(img_file), cv2.IMREAD_GRAYSCALE)
            if img is None:
                try:
                    img = np.array(Image.open(img_file).convert('L'))
                except:
                    continue
            
            h, w = img.shape[:2]
            if w != expected_width or h != expected_height:
                warnings.warn(f"Image size mismatch for {conn_id}: expected {expected_width}x{expected_height}, "
                           f"found {w}x{h} in {img_file.name}. Skipping connector.")
                size_mismatch = True
                break
        
        if size_mismatch:
            continue
        
        # Build mask - prefer polygon over bbox
        mask = None
        if 'polygon' in conn_config or 'points' in conn_config:
            points = conn_config.get('polygon') or conn_config.get('points')
            mask = create_mask_from_polygon(points, expected_width, expected_height)
        elif 'bbox' in conn_config:
            mask = create_mask_from_bbox(conn_config['bbox'], expected_width, expected_height)
        else:
            warnings.warn(f"No mask definition found for {conn_id}, skipping.")
            continue
        
        connector_info[conn_id] = {
            'mask': mask,
            'image_size': (expected_width, expected_height),
            'inpaint_radius': conn_config.get('inpaint_radius', DEFAULT_INPAINT_RADIUS),
            'mask_dilate_px': conn_config.get('mask_dilate_px', DEFAULT_MASK_DILATE_PX),
            'occ_weight_bbox': conn_config.get('occ_weight_bbox', None)
        }
    
    return connector_info

# Load config and build masks
print("Loading mask configuration...")
mask_config = load_mask_config(MASK_CONFIG_PATH)
connectors = get_connector_directories(CONNECTORS_ROOT)
print(f"Found connectors: {connectors}")

connector_masks = build_connector_masks(mask_config, connectors, CONNECTORS_ROOT)

# Summary table
print("\n" + "="*60)
print("Connector Mask Summary")
print("="*60)
print(f"{'Connector':<12} {'Image Size':<15} {'Mask Area':<12} {'Status':<10}")
print("-"*60)
for conn_id, info in connector_masks.items():
    mask_area = np.sum(info['mask'] > 0)
    total_pixels = info['mask'].size
    area_pct = (mask_area / total_pixels) * 100
    img_size = f"{info['image_size'][0]}x{info['image_size'][1]}"
    print(f"{conn_id:<12} {img_size:<15} {area_pct:>6.2f}%     {'OK':<10}")
print("="*60)


## 3. Visual Debug: Mask Overlay


In [ ]:
def overlay_mask_on_image(image: np.ndarray, mask: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    """Overlay mask on image with transparency."""
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    
    overlay = image.copy()
    mask_colored = np.zeros_like(image)
    mask_colored[mask > 0] = [255, 0, 0]  # Red overlay
    
    cv2.addWeighted(overlay, 1-alpha, mask_colored, alpha, 0, overlay)
    return overlay

def visualize_masks(connector_masks: Dict, connectors_root: Path, n_samples: int = 3):
    """Visualize mask overlays for sample images."""
    for conn_id, info in list(connector_masks.items())[:3]:  # Show first 3 connectors
        conn_dir = connectors_root / conn_id
        img_files = sorted(list(conn_dir.glob("*.png")) + list(conn_dir.glob("*.jpg")))
        
        if len(img_files) < n_samples:
            n_samples = len(img_files)
        
        selected_files = random.sample(img_files, min(n_samples, len(img_files)))
        
        fig, axes = plt.subplots(1, n_samples, figsize=(5*n_samples, 5))
        if n_samples == 1:
            axes = [axes]
        
        for idx, img_path in enumerate(selected_files):
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            
            overlay = overlay_mask_on_image(img, info['mask'])
            axes[idx].imshow(overlay)
            axes[idx].set_title(f"{conn_id}\n{img_path.name}")
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()

visualize_masks(connector_masks, CONNECTORS_ROOT, n_samples=3)


## 4. Visual Debug: Fake-KO Preview


In [ ]:
def generate_fake_ko(
    image: np.ndarray,
    mask: np.ndarray,
    inpaint_radius: int = DEFAULT_INPAINT_RADIUS,
    mask_dilate_px: int = DEFAULT_MASK_DILATE_PX,
    inpaint_method: int = cv2.INPAINT_TELEA,
    use_blending: bool = True
) -> np.ndarray:
    """
    Generate fake KO image by inpainting the PCL region.
    
    Args:
        image: Input grayscale or RGB image
        mask: Binary mask (255 where PCL is)
        inpaint_radius: Radius for inpainting
        mask_dilate_px: Pixels to dilate mask
        inpaint_method: cv2.INPAINT_TELEA or cv2.INPAINT_NS
        use_blending: Whether to apply boundary blending
    
    Returns:
        Fake KO image
    """
    # Convert to grayscale for processing
    if len(image.shape) == 3:
        img_gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        img_gray = image.copy()
    
    # Dilate mask
    if mask_dilate_px > 0:
        kernel = np.ones((mask_dilate_px*2+1, mask_dilate_px*2+1), np.uint8)
        mask_dilated = cv2.dilate(mask, kernel, iterations=1)
    else:
        mask_dilated = mask.copy()
    
    # Inpaint
    mask_inpaint = (mask_dilated > 0).astype(np.uint8) * 255
    inpainted = cv2.inpaint(img_gray, mask_inpaint, inpaint_radius, inpaint_method)
    
    # Boundary blending
    if use_blending:
        # Create alpha mask with Gaussian blur for smooth transition
        alpha_mask = mask_dilated.astype(np.float32) / 255.0
        alpha_mask = cv2.GaussianBlur(alpha_mask, (15, 15), 5)
        
        # Blend: alpha=1 in center (inpainted), alpha=0 outside (original)
        fake_ko = (alpha_mask * inpainted + (1 - alpha_mask) * img_gray).astype(np.uint8)
    else:
        fake_ko = inpainted
    
    # Convert back to original format if needed
    if len(image.shape) == 3:
        fake_ko = cv2.cvtColor(fake_ko, cv2.COLOR_GRAY2RGB)
    
    return fake_ko

def preview_fake_ko(connector_masks: Dict, connectors_root: Path, n_samples: int = 3):
    """Preview fake-KO generation for sample images."""
    conn_id = 'conn1'  # Focus on conn1 as specified
    if conn_id not in connector_masks:
        print(f"{conn_id} not found in connector_masks")
        return
    
    info = connector_masks[conn_id]
    conn_dir = connectors_root / conn_id
    img_files = sorted(list(conn_dir.glob("*.png")) + list(conn_dir.glob("*.jpg")))
    
    selected_files = random.sample(img_files, min(n_samples, len(img_files)))
    
    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4*n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, -1)
    
    for idx, img_path in enumerate(selected_files):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        
        fake_ko = generate_fake_ko(
            img,
            info['mask'],
            inpaint_radius=info['inpaint_radius'],
            mask_dilate_px=info['mask_dilate_px']
        )
        
        diff = np.abs(img.astype(np.float32) - fake_ko.astype(np.float32))
        diff = (diff / diff.max() * 255).astype(np.uint8) if diff.max() > 0 else diff.astype(np.uint8)
        
        axes[idx, 0].imshow(img, cmap='gray')
        axes[idx, 0].set_title(f"OK\n{img_path.name}")
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(fake_ko, cmap='gray')
        axes[idx, 1].set_title("Fake KO")
        axes[idx, 1].axis('off')
        
        axes[idx, 2].imshow(diff, cmap='hot')
        axes[idx, 2].set_title("Difference")
        axes[idx, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

preview_fake_ko(connector_masks, CONNECTORS_ROOT, n_samples=3)


## 5. PyTorch Dataset & Dataloader


In [ ]:
class AugmentationTransform:
    """Light augmentations for both OK and fake-KO images."""
    
    def __init__(self, brightness_range: Tuple[float, float] = (0.9, 1.1),
                 contrast_range: Tuple[float, float] = (0.9, 1.1),
                 noise_std: float = 5.0,
                 blur_prob: float = 0.1):
        self.brightness_range = brightness_range
        self.contrast_range = contrast_range
        self.noise_std = noise_std
        self.blur_prob = blur_prob
    
    def __call__(self, image: np.ndarray) -> np.ndarray:
        """Apply random augmentations."""
        img = image.copy().astype(np.float32)
        
        # Brightness jitter
        brightness_factor = np.random.uniform(*self.brightness_range)
        img = img * brightness_factor
        
        # Contrast jitter
        contrast_factor = np.random.uniform(*self.contrast_range)
        mean = img.mean()
        img = (img - mean) * contrast_factor + mean
        
        # Gaussian noise
        noise = np.random.normal(0, self.noise_std, img.shape)
        img = img + noise
        
        # Optional slight blur
        if np.random.random() < self.blur_prob:
            img = cv2.GaussianBlur(img.astype(np.uint8), (3, 3), 0.5).astype(np.float32)
        
        # Clip to valid range
        img = np.clip(img, 0, 255).astype(np.uint8)
        return img

class PCLPresenceDataset(Dataset):
    """Dataset that generates fake-KO on-the-fly."""
    
    def __init__(
        self,
        image_paths: List[Path],
        mask: np.ndarray,
        inpaint_radius: int,
        mask_dilate_px: int,
        is_training: bool = True,
        use_augmentation: bool = True,
        grayscale_mode: bool = True
    ):
        self.image_paths = image_paths
        self.mask = mask
        self.inpaint_radius = inpaint_radius
        self.mask_dilate_px = mask_dilate_px
        self.is_training = is_training
        self.use_augmentation = use_augmentation
        self.grayscale_mode = grayscale_mode
        
        if use_augmentation:
            self.aug_transform = AugmentationTransform()
        
        # For diversity across epochs
        self.epoch = 0
    
    def set_epoch(self, epoch: int):
        """Set epoch for diverse fake-KO generation."""
        self.epoch = epoch
    
    def __len__(self) -> int:
        # Return double length: OK + fake-KO pairs
        return len(self.image_paths) * 2
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        # Determine if this is OK (label 0) or fake-KO (label 1)
        is_fake_ko = idx % 2 == 1
        img_idx = idx // 2
        
        # Load image
        img_path = self.image_paths[img_idx]
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE if self.grayscale_mode else cv2.IMREAD_COLOR)
        
        if img is None:
            # Fallback: try PIL
            img = np.array(Image.open(img_path).convert('L' if self.grayscale_mode else 'RGB'))
        
        if is_fake_ko:
            # Generate fake-KO with diversity
            inpaint_method = cv2.INPAINT_TELEA if np.random.random() < 0.8 else cv2.INPAINT_NS
            radius_variation = np.random.uniform(0.8, 1.2)
            inpaint_radius = int(self.inpaint_radius * radius_variation)
            dilate_variation = np.random.randint(-1, 2)
            mask_dilate = max(0, self.mask_dilate_px + dilate_variation)
            
            img = generate_fake_ko(
                img, self.mask,
                inpaint_radius=inpaint_radius,
                mask_dilate_px=mask_dilate,
                inpaint_method=inpaint_method,
                use_blending=True
            )
            label = 1
        else:
            label = 0
        
        # Apply augmentations
        if self.use_augmentation and self.is_training:
            img = self.aug_transform(img)
        
        # Convert to tensor
        if self.grayscale_mode:
            if len(img.shape) == 2:
                img = img[np.newaxis, :, :]  # Add channel dimension
            else:
                img = img[:, :, 0:1].transpose(2, 0, 1)  # Take first channel
        else:
            if len(img.shape) == 2:
                img = np.stack([img, img, img], axis=0)  # Replicate to 3 channels
            else:
                img = img.transpose(2, 0, 1)
        
        img_tensor = torch.from_numpy(img).float() / 255.0
        
        return img_tensor, label

def create_dataloaders(
    connector_id: str,
    connector_info: Dict,
    connectors_root: Path,
    train_val_split: float = DEFAULT_TRAIN_VAL_SPLIT,
    batch_size: int = DEFAULT_BATCH_SIZE,
    num_workers: int = 2
) -> Tuple[DataLoader, DataLoader]:
    """Create train and validation dataloaders."""
    conn_dir = connectors_root / connector_id
    img_files = sorted(list(conn_dir.glob("*.png")) + list(conn_dir.glob("*.jpg")))
    
    if not img_files:
        raise ValueError(f"No images found for {connector_id}")
    
    # Split into train/val
    split_idx = int(len(img_files) * train_val_split)
    train_files = img_files[:split_idx]
    val_files = img_files[split_idx:]
    
    print(f"  Train images: {len(train_files)}, Val images: {len(val_files)}")
    
    # Create datasets
    train_dataset = PCLPresenceDataset(
        train_files,
        connector_info['mask'],
        connector_info['inpaint_radius'],
        connector_info['mask_dilate_px'],
        is_training=True,
        use_augmentation=True
    )
    
    val_dataset = PCLPresenceDataset(
        val_files,
        connector_info['mask'],
        connector_info['inpaint_radius'],
        connector_info['mask_dilate_px'],
        is_training=False,
        use_augmentation=False
    )
    
    # Create dataloaders with balanced sampling
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    return train_loader, val_loader


## 6. Model Definition


## Debug & Visualization (Presentation)

This section provides visual explanations and analysis of the trained models.


In [ ]:
# CELL A — Connector overview + examples
connector_id = "conn1"  # Change this to visualize different connectors

conn_dir = CONNECTORS_ROOT / connector_id
img_files = sorted(list(conn_dir.glob("*.png")) + list(conn_dir.glob("*.jpg")))

if connector_id not in connector_masks:
    print(f"Error: {connector_id} not found in connector_masks")
else:
    info = connector_masks[connector_id]
    mask = info['mask']
    img_size = info['image_size']
    
    # Calculate mask area percentage
    mask_area = np.sum(mask > 0)
    total_pixels = mask.size
    area_pct = (mask_area / total_pixels) * 100
    
    print(f"Connector: {connector_id}")
    print(f"Number of images: {len(img_files)}")
    print(f"Image size: {img_size[0]}x{img_size[1]}")
    print(f"Mask area: {area_pct:.2f}%")
    print("="*60)
    
    # Show grid of 6 random OK images with mask overlay
    n_samples = min(6, len(img_files))
    selected_files = random.sample(img_files, n_samples)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(selected_files):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.array(Image.open(img_path).convert('L'))
        
        # Overlay mask
        overlay = overlay_mask_on_image(img, mask, alpha=0.5)
        
        axes[idx].imshow(overlay)
        axes[idx].set_title(f"{img_path.name}", fontsize=10, weight='bold')
        axes[idx].axis('off')
    
    plt.suptitle(f"{connector_id.upper()} - Sample Images with Mask Overlay", 
                 fontsize=14, weight='bold', y=0.995)
    plt.tight_layout()
    plt.show()


In [ ]:
# CELL B — Fake-KO generation demo
connector_id = "conn1"  # Change this to visualize different connectors

if connector_id not in connector_masks:
    print(f"Error: {connector_id} not found in connector_masks")
else:
    info = connector_masks[connector_id]
    conn_dir = CONNECTORS_ROOT / connector_id
    img_files = sorted(list(conn_dir.glob("*.png")) + list(conn_dir.glob("*.jpg")))
    
    # Pick 3 random OK images
    n_samples = min(3, len(img_files))
    selected_files = random.sample(img_files, n_samples)
    
    fig, axes = plt.subplots(n_samples, 3, figsize=(15, 5*n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, -1)
    
    for idx, img_path in enumerate(selected_files):
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.array(Image.open(img_path).convert('L'))
        
        # Generate fake KO
        fake_ko = generate_fake_ko(
            img,
            info['mask'],
            inpaint_radius=info['inpaint_radius'],
            mask_dilate_px=info['mask_dilate_px']
        )
        
        # Difference heatmap
        diff = np.abs(img.astype(np.float32) - fake_ko.astype(np.float32))
        diff_normalized = (diff / diff.max() * 255).astype(np.uint8) if diff.max() > 0 else diff.astype(np.uint8)
        
        # Display
        axes[idx, 0].imshow(img, cmap='gray')
        axes[idx, 0].set_title(f"OK\n{img_path.name}", fontsize=11, weight='bold')
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(fake_ko, cmap='gray')
        axes[idx, 1].set_title("Fake KO", fontsize=11, weight='bold', color='red')
        axes[idx, 1].axis('off')
        
        axes[idx, 2].imshow(diff_normalized, cmap='hot')
        axes[idx, 2].set_title("Difference", fontsize=11, weight='bold')
        axes[idx, 2].axis('off')
    
    plt.suptitle(f"{connector_id.upper()} - Fake-KO Generation Demo", 
                 fontsize=14, weight='bold', y=0.995)
    plt.tight_layout()
    plt.show()


In [ ]:
# CELL C — Real OK vs Real KO gallery (if Test folder exists)
connector_id = "conn1"  # Change this to visualize different connectors

# Check for Test folder
test_folder = CONNECTORS_ROOT / connector_id / "Test"
if not test_folder.exists():
    print(f"No Test folder found for {connector_id}")
    print("Skipping real OK vs KO gallery visualization")
else:
    # Load model and threshold
    try:
        model, threshold = load_trained_model(connector_id, OUTPUT_ROOT, device)
        info = connector_masks[connector_id]
        
        # Get all images from Test folder
        test_images = sorted(list(test_folder.glob("*.png")) + list(test_folder.glob("*.jpg")))
        
        if not test_images:
            print(f"No images found in Test folder for {connector_id}")
        else:
            # Run inference for all images
            print(f"Running inference on {len(test_images)} test images...")
            results = []
            
            for img_path in tqdm(test_images):
                prob_ko, is_ko = predict_connector(
                    img_path, connector_id, model, info, threshold, device
                )
                results.append({
                    'path': img_path,
                    'prob_ko': prob_ko,
                    'is_ko': is_ko
                })
            
            # Sort by prob_ko
            results_sorted = sorted(results, key=lambda x: x['prob_ko'])
            
            # Get 10 highest (KO) and 10 lowest (OK)
            lowest_10 = results_sorted[:10]
            highest_10 = results_sorted[-10:]
            
            # Display gallery
            fig, axes = plt.subplots(2, 10, figsize=(20, 4))
            
            # Lowest 10 (OK predictions)
            for idx, result in enumerate(lowest_10):
                img = cv2.imread(str(result['path']), cv2.IMREAD_GRAYSCALE)
                if img is None:
                    img = np.array(Image.open(result['path']).convert('L'))
                
                axes[0, idx].imshow(img, cmap='gray')
                axes[0, idx].set_title(f"{result['path'].name[:15]}...\np_ko={result['prob_ko']:.3f}", 
                                       fontsize=8, color='green')
                axes[0, idx].axis('off')
            
            axes[0, 0].set_ylabel("Lowest prob_ko (OK)", fontsize=12, weight='bold', color='green')
            
            # Highest 10 (KO predictions)
            for idx, result in enumerate(highest_10):
                img = cv2.imread(str(result['path']), cv2.IMREAD_GRAYSCALE)
                if img is None:
                    img = np.array(Image.open(result['path']).convert('L'))
                
                axes[1, idx].imshow(img, cmap='gray')
                axes[1, idx].set_title(f"{result['path'].name[:15]}...\np_ko={result['prob_ko']:.3f}", 
                                       fontsize=8, color='red')
                axes[1, idx].axis('off')
            
            axes[1, 0].set_ylabel("Highest prob_ko (KO)", fontsize=12, weight='bold', color='red')
            
            plt.suptitle(f"{connector_id.upper()} - Real OK vs Real KO Gallery (Test Set)", 
                         fontsize=14, weight='bold')
            plt.tight_layout()
            plt.show()
            
    except Exception as e:
        print(f"Error loading model or processing Test folder: {e}")
        import traceback
        traceback.print_exc()


In [ ]:
# CELL D — Score distribution plot + threshold
connector_id = "conn1"  # Change this to visualize different connectors

try:
    # Load model and threshold
    model, threshold = load_trained_model(connector_id, OUTPUT_ROOT, device)
    info = connector_masks[connector_id]
    
    # Get images from Data/connectors/<connector_id>/ and optionally Test/
    conn_dir = CONNECTORS_ROOT / connector_id
    img_files = sorted(list(conn_dir.glob("*.png")) + list(conn_dir.glob("*.jpg")))
    
    # Optionally add Test folder images
    test_folder = conn_dir / "Test"
    if test_folder.exists():
        test_images = sorted(list(test_folder.glob("*.png")) + list(test_folder.glob("*.jpg")))
        img_files.extend(test_images)
    
    print(f"Computing prob_ko for {len(img_files)} images...")
    
    # Compute prob_ko for all images
    all_probs = []
    all_predictions = []
    
    for img_path in tqdm(img_files):
        prob_ko, is_ko = predict_connector(
            img_path, connector_id, model, info, threshold, device
        )
        all_probs.append(prob_ko)
        all_predictions.append('KO' if is_ko else 'OK')
    
    all_probs = np.array(all_probs)
    all_predictions = np.array(all_predictions)
    
    # Separate OK and KO predictions
    ok_probs = all_probs[all_predictions == 'OK']
    ko_probs = all_probs[all_predictions == 'KO']
    
    # Plot histogram
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    
    if len(ok_probs) > 0:
        ax.hist(ok_probs, bins=50, alpha=0.7, label=f'Predicted OK (n={len(ok_probs)})', 
                color='green', edgecolor='black')
    
    if len(ko_probs) > 0:
        ax.hist(ko_probs, bins=50, alpha=0.7, label=f'Predicted KO (n={len(ko_probs)})', 
                color='red', edgecolor='black')
    
    # Draw threshold line
    ax.axvline(threshold, color='blue', linestyle='--', linewidth=2, 
               label=f'Threshold = {threshold:.4f}')
    
    ax.set_xlabel('Probability KO', fontsize=12, weight='bold')
    ax.set_ylabel('Frequency', fontsize=12, weight='bold')
    ax.set_title(f'{connector_id.upper()} - Score Distribution', fontsize=14, weight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*60)
    print("Score Distribution Summary")
    print("="*60)
    
    if len(ok_probs) > 0:
        print(f"Predicted OK:")
        print(f"  Min:   {np.min(ok_probs):.4f}")
        print(f"  Mean:  {np.mean(ok_probs):.4f}")
        print(f"  Max:   {np.max(ok_probs):.4f}")
    
    if len(ko_probs) > 0:
        print(f"\nPredicted KO:")
        print(f"  Min:   {np.min(ko_probs):.4f}")
        print(f"  Mean:  {np.mean(ko_probs):.4f}")
        print(f"  Max:   {np.max(ko_probs):.4f}")
    
    if len(ok_probs) > 0 and len(ko_probs) > 0:
        gap = np.min(ko_probs) - np.max(ok_probs)
        print(f"\nGAP (minKO - maxOK): {gap:.4f}")
        if gap > 0:
            print("✅ Good separation between OK and KO")
        else:
            print("⚠️  Overlap between OK and KO distributions")
    
    print(f"\nThreshold: {threshold:.4f}")
    print("="*60)
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# CELL E — Borderline cases
connector_id = "conn1"  # Change this to visualize different connectors

try:
    # Load model and threshold
    model, threshold = load_trained_model(connector_id, OUTPUT_ROOT, device)
    info = connector_masks[connector_id]
    
    # Get images from Data/connectors/<connector_id>/ and optionally Test/
    conn_dir = CONNECTORS_ROOT / connector_id
    img_files = sorted(list(conn_dir.glob("*.png")) + list(conn_dir.glob("*.jpg")))
    
    test_folder = conn_dir / "Test"
    if test_folder.exists():
        test_images = sorted(list(test_folder.glob("*.png")) + list(test_folder.glob("*.jpg")))
        img_files.extend(test_images)
    
    print(f"Finding borderline cases from {len(img_files)} images...")
    
    # Compute prob_ko and distance from threshold
    borderline_data = []
    
    for img_path in tqdm(img_files):
        prob_ko, is_ko = predict_connector(
            img_path, connector_id, model, info, threshold, device
        )
        distance_from_threshold = abs(prob_ko - threshold)
        borderline_data.append({
            'path': img_path,
            'prob_ko': prob_ko,
            'distance': distance_from_threshold
        })
    
    # Sort by distance from threshold (closest first)
    borderline_data.sort(key=lambda x: x['distance'])
    
    # Get 12 closest to threshold
    n_borderline = min(12, len(borderline_data))
    borderline_samples = borderline_data[:n_borderline]
    
    # Display as grid
    n_cols = 4
    n_rows = (n_borderline + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    axes = axes.flatten()
    
    for idx, sample in enumerate(borderline_samples):
        img = cv2.imread(str(sample['path']), cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.array(Image.open(sample['path']).convert('L'))
        
        axes[idx].imshow(img, cmap='gray')
        
        # Color based on prediction
        pred_color = 'red' if sample['prob_ko'] > threshold else 'green'
        title = f"{sample['path'].name[:20]}...\np_ko={sample['prob_ko']:.4f}\ndist={sample['distance']:.4f}"
        axes[idx].set_title(title, fontsize=9, color=pred_color, weight='bold')
        axes[idx].axis('off')
    
    # Hide unused axes
    for idx in range(n_borderline, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle(f'{connector_id.upper()} - Borderline Cases (Closest to Threshold)', 
                 fontsize=14, weight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"\nShowing {n_borderline} samples closest to threshold {threshold:.4f}")
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# CELL F — Grad-CAM / Saliency visualization
connector_id = "conn1"  # Change this to visualize different connectors

try:
    # Load model
    model, threshold = load_trained_model(connector_id, OUTPUT_ROOT, device)
    info = connector_masks[connector_id]
    
    # Get test images (or use connector images)
    conn_dir = CONNECTORS_ROOT / connector_id
    test_folder = conn_dir / "Test"
    
    if test_folder.exists():
        test_images = sorted(list(test_folder.glob("*.png")) + list(test_folder.glob("*.jpg")))
    else:
        test_images = sorted(list(conn_dir.glob("*.png")) + list(conn_dir.glob("*.jpg")))
    
    if not test_images:
        print(f"No images found for {connector_id}")
    else:
        # Run inference to get OK and KO samples
        results = []
        for img_path in test_images[:50]:  # Limit to first 50 for speed
            prob_ko, is_ko = predict_connector(
                img_path, connector_id, model, info, threshold, device
            )
            results.append({
                'path': img_path,
                'prob_ko': prob_ko,
                'is_ko': is_ko
            })
        
        # Sort and get 3 OK and 3 KO
        results_sorted = sorted(results, key=lambda x: x['prob_ko'])
        ok_samples = [r for r in results_sorted if not r['is_ko']][:3]
        ko_samples = [r for r in results_sorted if r['is_ko']][-3:]
        
        if len(ok_samples) == 0 or len(ko_samples) == 0:
            print("Need both OK and KO samples for visualization")
        else:
            # Implement Grad-CAM
            def generate_gradcam(model, image_tensor, target_layer):
                """Generate Grad-CAM heatmap."""
                model.eval()
                
                # Register hook for gradients
                gradients = []
                activations = []
                
                def backward_hook(module, grad_input, grad_output):
                    gradients.append(grad_output[0])
                
                def forward_hook(module, input, output):
                    activations.append(output)
                
                hook_handle = target_layer.register_backward_hook(backward_hook)
                forward_handle = target_layer.register_forward_hook(forward_hook)
                
                # Forward pass
                output = model(image_tensor)
                
                # Backward pass
                model.zero_grad()
                output.backward(torch.ones_like(output))
                
                # Get gradients and activations
                grads = gradients[0]
                acts = activations[0]
                
                # Global average pooling of gradients
                weights = torch.mean(grads, dim=(2, 3), keepdim=True)
                
                # Weighted combination of activation maps
                cam = torch.sum(weights * acts, dim=1, keepdim=True)
                cam = torch.relu(cam)
                
                # Normalize
                cam = cam.squeeze().cpu().numpy()
                cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                
                # Resize to input size
                cam_resized = cv2.resize(cam, (image_tensor.shape[3], image_tensor.shape[2]))
                
                hook_handle.remove()
                forward_handle.remove()
                
                return cam_resized
            
            # Get target layer (layer4 of ResNet18)
            target_layer = model.backbone.layer4
            
            # Visualize
            fig, axes = plt.subplots(2, 6, figsize=(18, 6))
            
            # OK samples
            for idx, sample in enumerate(ok_samples):
                img_path = sample['path']
                img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                if img is None:
                    img = np.array(Image.open(img_path).convert('L'))
                
                # Preprocess for model
                img_tensor = torch.from_numpy(img).float().unsqueeze(0).unsqueeze(0) / 255.0
                img_tensor = img_tensor.to(device)
                
                # Generate Grad-CAM
                try:
                    cam = generate_gradcam(model, img_tensor, target_layer)
                    
                    # Overlay heatmap
                    cam_colored = plt.cm.jet(cam)[:, :, :3]
                    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) / 255.0
                    overlay = 0.6 * img_rgb + 0.4 * cam_colored
                    
                    axes[0, idx].imshow(img, cmap='gray')
                    axes[0, idx].set_title(f"OK\n{sample['path'].name[:15]}...\np={sample['prob_ko']:.3f}", 
                                           fontsize=9, color='green', weight='bold')
                    axes[0, idx].axis('off')
                    
                    axes[1, idx].imshow(overlay)
                    axes[1, idx].set_title("Grad-CAM", fontsize=9, weight='bold')
                    axes[1, idx].axis('off')
                except Exception as e:
                    print(f"Error generating Grad-CAM for {img_path.name}: {e}")
                    axes[0, idx].text(0.5, 0.5, "Error", ha='center', va='center')
                    axes[1, idx].text(0.5, 0.5, "Error", ha='center', va='center')
            
            # KO samples
            for idx, sample in enumerate(ko_samples):
                img_path = sample['path']
                img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                if img is None:
                    img = np.array(Image.open(img_path).convert('L'))
                
                # Preprocess for model
                img_tensor = torch.from_numpy(img).float().unsqueeze(0).unsqueeze(0) / 255.0
                img_tensor = img_tensor.to(device)
                
                # Generate Grad-CAM
                try:
                    cam = generate_gradcam(model, img_tensor, target_layer)
                    
                    # Overlay heatmap
                    cam_colored = plt.cm.jet(cam)[:, :, :3]
                    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB) / 255.0
                    overlay = 0.6 * img_rgb + 0.4 * cam_colored
                    
                    axes[0, idx+3].imshow(img, cmap='gray')
                    axes[0, idx+3].set_title(f"KO\n{sample['path'].name[:15]}...\np={sample['prob_ko']:.3f}", 
                                             fontsize=9, color='red', weight='bold')
                    axes[0, idx+3].axis('off')
                    
                    axes[1, idx+3].imshow(overlay)
                    axes[1, idx+3].set_title("Grad-CAM", fontsize=9, weight='bold')
                    axes[1, idx+3].axis('off')
                except Exception as e:
                    print(f"Error generating Grad-CAM for {img_path.name}: {e}")
                    axes[0, idx+3].text(0.5, 0.5, "Error", ha='center', va='center')
                    axes[1, idx+3].text(0.5, 0.5, "Error", ha='center', va='center')
            
            plt.suptitle(f'{connector_id.upper()} - Grad-CAM Visualization (Layer4)', 
                        fontsize=14, weight='bold')
            plt.tight_layout()
            plt.show()
            
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# CELL G — Multi-connector summary visualization
print("="*80)
print("Multi-Connector Summary")
print("="*80)

# Scan outputs/pcl_presence/*
summary_data = []

for conn_dir in OUTPUT_ROOT.iterdir():
    if not conn_dir.is_dir():
        continue
    
    connector_id = conn_dir.name
    
    # Load threshold
    threshold_path = conn_dir / "threshold.json"
    if not threshold_path.exists():
        continue
    
    with open(threshold_path, 'r') as f:
        threshold_data = json.load(f)
    threshold = threshold_data['threshold']
    
    # Count images
    conn_data_dir = CONNECTORS_ROOT / connector_id
    if conn_data_dir.exists():
        img_files = list(conn_data_dir.glob("*.png")) + list(conn_data_dir.glob("*.jpg"))
        n_images = len(img_files)
    else:
        n_images = 0
    
    # Load config for notes
    config_path = conn_dir / "training_config.json"
    notes = ""
    if config_path.exists():
        with open(config_path, 'r') as f:
            config = json.load(f)
            notes = f"AUC={config.get('final_val_auc', 'N/A'):.3f}" if config.get('final_val_auc') else ""
    
    summary_data.append({
        'connector_id': connector_id,
        'threshold': threshold,
        'n_images': n_images,
        'notes': notes
    })

# Sort by connector number
summary_data.sort(key=lambda x: int(x['connector_id'][4:]) if x['connector_id'][4:].isdigit() else 999)

# Print table
print(f"{'Connector':<12} {'Threshold':<12} {'# Images':<10} {'Notes':<20}")
print("-"*80)

for data in summary_data:
    threshold_str = f"{data['threshold']:.4f}"
    print(f"{data['connector_id']:<12} {threshold_str:<12} {data['n_images']:<10} {data['notes']:<20}")

print("="*80)

# Plot thresholds as bar plot
if len(summary_data) > 0:
    connectors = [d['connector_id'] for d in summary_data]
    thresholds = [d['threshold'] for d in summary_data]
    
    fig, ax = plt.subplots(1, 1, figsize=(max(10, len(connectors)*1.2), 6))
    
    bars = ax.bar(connectors, thresholds, color='steelblue', edgecolor='black', linewidth=1.5)
    
    # Add value labels on bars
    for bar, thresh in zip(bars, thresholds):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{thresh:.4f}',
               ha='center', va='bottom', fontsize=9, weight='bold')
    
    ax.set_xlabel('Connector', fontsize=12, weight='bold')
    ax.set_ylabel('Threshold', fontsize=12, weight='bold')
    ax.set_title('Thresholds Across Connectors', fontsize=14, weight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    print(f"\nTotal connectors trained: {len(summary_data)}")
else:
    print("\nNo trained connectors found in outputs/pcl_presence/")


In [ ]:
class PCLPresenceClassifier(nn.Module):
    """ResNet18-based binary classifier for PCL presence detection."""
    
    def __init__(self, input_channels: int = 1, pretrained: bool = True):
        super().__init__()
        # Load pretrained ResNet18 (support both old and new PyTorch versions)
        try:
            # New API (torchvision >= 0.13)
            from torchvision.models import ResNet18_Weights
            weights = ResNet18_Weights.DEFAULT if pretrained else None
            self.backbone = resnet18(weights=weights)
        except (ImportError, AttributeError):
            # Old API (torchvision < 0.13)
            self.backbone = resnet18(pretrained=pretrained)
        
        # Modify first layer for grayscale input
        if input_channels == 1:
            # Replace first conv layer
            self.backbone.conv1 = nn.Conv2d(
                1, 64, kernel_size=7, stride=2, padding=3, bias=False
            )
        elif input_channels == 3:
            # Keep original (already 3 channels)
            pass
        else:
            raise ValueError(f"Unsupported input_channels: {input_channels}")
        
        # Replace classifier head for binary classification
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, 1)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)

def create_model(input_channels: int = 1, pretrained: bool = True) -> nn.Module:
    """Create and return the model."""
    model = PCLPresenceClassifier(input_channels=input_channels, pretrained=pretrained)
    return model

# Test model creation
test_model = create_model(input_channels=1, pretrained=True)
print("Model architecture:")
print(test_model)

# Count parameters
total_params = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


## 7. Training Loop


In [ ]:
def train_epoch(model: nn.Module, train_loader: DataLoader, criterion: nn.Module,
                optimizer: optim.Optimizer, device: torch.device) -> float:
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for images, labels in tqdm(train_loader, desc="Training"):
        images = images.to(device)
        labels = labels.float().to(device)
        
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches if num_batches > 0 else 0.0

def validate(model: nn.Module, val_loader: DataLoader, criterion: nn.Module,
             device: torch.device) -> Tuple[float, List[float], List[int]]:
    """Validate model and return loss, predictions, and labels."""
    model.eval()
    total_loss = 0.0
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validating"):
            images = images.to(device)
            labels = labels.float().to(device)
            
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())
            
            total_loss += loss.item()
    
    avg_loss = total_loss / len(val_loader) if len(val_loader) > 0 else 0.0
    return avg_loss, all_probs, all_labels

def train_model(
    connector_id: str,
    connector_info: Dict,
    connectors_root: Path,
    epochs: int = DEFAULT_EPOCHS,
    batch_size: int = DEFAULT_BATCH_SIZE,
    learning_rate: float = DEFAULT_LEARNING_RATE,
    train_val_split: float = DEFAULT_TRAIN_VAL_SPLIT
) -> Tuple[nn.Module, Dict]:
    """Train model for a single connector."""
    print(f"\n{'='*60}")
    print(f"Training model for {connector_id}")
    print(f"{'='*60}")
    
    # Create dataloaders
    train_loader, val_loader = create_dataloaders(
        connector_id, connector_info, connectors_root,
        train_val_split=train_val_split,
        batch_size=batch_size
    )
    
    # Create model
    model = create_model(input_channels=1, pretrained=True)
    model = model.to(device)
    
    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_auc': []
    }
    
    best_val_loss = float('inf')
    patience = 5
    patience_counter = 0
    
    # Training loop
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        # Update dataset epoch for diversity
        train_loader.dataset.set_epoch(epoch)
        
        # Train
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        val_loss, val_probs, val_labels = validate(model, val_loader, criterion, device)
        
        # Calculate AUC
        try:
            val_auc = roc_auc_score(val_labels, val_probs)
        except:
            val_auc = 0.0
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)
        
        print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    return model, history


## 8. Threshold Calibration (Real OK Only)


In [ ]:
def calibrate_threshold(
    model: nn.Module,
    val_ok_images: List[Path],
    connector_info: Dict,
    device: torch.device,
    percentile: float = DEFAULT_THRESHOLD_PERCENTILE
) -> float:
    """
    Calibrate threshold using ONLY real OK validation images.
    Returns threshold at specified percentile of KO probabilities.
    """
    model.eval()
    ko_probs = []
    
    # Create a simple dataset for OK images only
    dataset = PCLPresenceDataset(
        val_ok_images,
        connector_info['mask'],
        connector_info['inpaint_radius'],
        connector_info['mask_dilate_px'],
        is_training=False,
        use_augmentation=False
    )
    
    with torch.no_grad():
        for idx in range(len(val_ok_images)):
            # Get OK image (even indices)
            image, _ = dataset[idx * 2]  # OK images are at even indices
            image = image.unsqueeze(0).to(device)
            
            output = model(image).squeeze()
            prob_ko = torch.sigmoid(output).item()
            ko_probs.append(prob_ko)
    
    threshold = np.percentile(ko_probs, percentile)
    false_positive_rate = np.mean(np.array(ko_probs) > threshold) * 100
    
    print(f"\nThreshold Calibration (Real OK Only):")
    print(f"  Percentile: {percentile}%")
    print(f"  Threshold: {threshold:.4f}")
    print(f"  Expected FPR on OK: {false_positive_rate:.2f}%")
    print(f"  Min KO prob: {np.min(ko_probs):.4f}")
    print(f"  Max KO prob: {np.max(ko_probs):.4f}")
    print(f"  Mean KO prob: {np.mean(ko_probs):.4f}")
    
    return threshold, false_positive_rate


## 9. Inference Helper


In [ ]:
def predict_connector(
    image_path: Union[str, Path],
    connector_id: str,
    model: nn.Module,
    connector_info: Dict,
    threshold: float,
    device: torch.device
) -> Tuple[float, bool]:
    """
    Predict PCL presence for a single image.
    
    Returns:
        (prob_ko, is_ko): KO probability and binary prediction
    """
    model.eval()
    
    # Load image
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        img = np.array(Image.open(image_path).convert('L'))
    
    # Preprocess
    if len(img.shape) == 2:
        img = img[np.newaxis, :, :]
    else:
        img = img[:, :, 0:1].transpose(2, 0, 1)
    
    img_tensor = torch.from_numpy(img).float().unsqueeze(0) / 255.0
    img_tensor = img_tensor.to(device)
    
    # Predict
    with torch.no_grad():
        output = model(img_tensor).squeeze()
        prob_ko = torch.sigmoid(output).item()
    
    is_ko = prob_ko > threshold
    
    return prob_ko, is_ko


## 10. Save Artifacts


In [ ]:
def save_artifacts(
    connector_id: str,
    model: nn.Module,
    threshold: float,
    fpr_estimate: float,
    history: Dict,
    connector_info: Dict,
    output_root: Path
):
    """Save model weights, threshold, and training config."""
    output_dir = output_root / connector_id
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save model weights
    model_path = output_dir / "model.pt"
    torch.save(model.state_dict(), model_path)
    print(f"Saved model to: {model_path}")
    
    # Save threshold
    threshold_data = {
        'threshold': float(threshold),
        'percentile': DEFAULT_THRESHOLD_PERCENTILE,
        'val_ok_fpr_estimate': float(fpr_estimate)
    }
    threshold_path = output_dir / "threshold.json"
    with open(threshold_path, 'w') as f:
        json.dump(threshold_data, f, indent=2)
    print(f"Saved threshold to: {threshold_path}")
    
    # Save training config summary
    config_summary = {
        'connector_id': connector_id,
        'image_size': connector_info['image_size'],
        'inpaint_radius': connector_info['inpaint_radius'],
        'mask_dilate_px': connector_info['mask_dilate_px'],
        'threshold': float(threshold),
        'val_ok_fpr_estimate': float(fpr_estimate),
        'final_train_loss': history['train_loss'][-1] if history['train_loss'] else None,
        'final_val_loss': history['val_loss'][-1] if history['val_loss'] else None,
        'final_val_auc': history['val_auc'][-1] if history['val_auc'] else None,
        'num_epochs_trained': len(history['train_loss'])
    }
    config_path = output_dir / "training_config.json"
    with open(config_path, 'w') as f:
        json.dump(config_summary, f, indent=2)
    print(f"Saved config to: {config_path}")


## 11. Multi-Connector Training Loop


In [ ]:
def train_all_connectors(
    connector_masks: Dict,
    connectors_root: Path,
    output_root: Path,
    epochs: int = DEFAULT_EPOCHS,
    batch_size: int = DEFAULT_BATCH_SIZE,
    learning_rate: float = DEFAULT_LEARNING_RATE
) -> List[Dict]:
    """Train models for all connectors and return summary."""
    results = []
    
    # Get all connector directories
    all_connectors = sorted([d.name for d in connectors_root.iterdir() 
                             if d.is_dir() and d.name.startswith('conn')])
    
    for conn_id in all_connectors:
        try:
            print(f"\n{'='*80}")
            print(f"Processing {conn_id}")
            print(f"{'='*80}")
            
            # Count images
            conn_dir = connectors_root / conn_id
            img_files = list(conn_dir.glob("*.png")) + list(conn_dir.glob("*.jpg"))
            n_images = len(img_files)
            
            # Check if connector has mask config
            if conn_id not in connector_masks:
                results.append({
                    'connector_id': conn_id,
                    'n_images': n_images,
                    'threshold': None,
                    'val_ok_fpr_estimate': None,
                    'status': 'SKIPPED',
                    'reason': 'No mask config or size mismatch'
                })
                print(f"Skipping {conn_id}: No mask config or image size mismatch")
                continue
            
            if n_images == 0:
                results.append({
                    'connector_id': conn_id,
                    'n_images': 0,
                    'threshold': None,
                    'val_ok_fpr_estimate': None,
                    'status': 'SKIPPED',
                    'reason': 'No images found'
                })
                print(f"Skipping {conn_id}: No images found")
                continue
            
            conn_info = connector_masks[conn_id]
            
            # Train model
            model, history = train_model(
                conn_id, conn_info, connectors_root,
                epochs=epochs,
                batch_size=batch_size,
                learning_rate=learning_rate
            )
            
            # Get validation OK images for threshold calibration
            img_files = sorted(list(conn_dir.glob("*.png")) + list(conn_dir.glob("*.jpg")))
            split_idx = int(len(img_files) * DEFAULT_TRAIN_VAL_SPLIT)
            val_ok_images = img_files[split_idx:]
            
            # Calibrate threshold
            threshold, fpr_estimate = calibrate_threshold(
                model, val_ok_images, conn_info, device,
                percentile=DEFAULT_THRESHOLD_PERCENTILE
            )
            
            # Save artifacts
            save_artifacts(conn_id, model, threshold, fpr_estimate, history, conn_info, output_root)
            
            results.append({
                'connector_id': conn_id,
                'n_images': n_images,
                'threshold': float(threshold),
                'val_ok_fpr_estimate': float(fpr_estimate),
                'status': 'TRAINED',
                'reason': None
            })
            
        except Exception as e:
            print(f"ERROR processing {conn_id}: {str(e)}")
            import traceback
            traceback.print_exc()
            results.append({
                'connector_id': conn_id,
                'n_images': n_images if 'n_images' in locals() else 0,
                'threshold': None,
                'val_ok_fpr_estimate': None,
                'status': 'ERROR',
                'reason': str(e)[:50]  # Truncate long error messages
            })
    
    return results

# Run training for all connectors
print("\n" + "="*80)
print("STARTING MULTI-CONNECTOR TRAINING")
print("="*80)

training_results = train_all_connectors(
    connector_masks,
    CONNECTORS_ROOT,
    OUTPUT_ROOT,
    epochs=DEFAULT_EPOCHS,
    batch_size=DEFAULT_BATCH_SIZE,
    learning_rate=DEFAULT_LEARNING_RATE
)

# Print final summary table
print("\n" + "="*100)
print("FINAL TRAINING SUMMARY")
print("="*100)
print(f"{'Connector':<12} {'N Images':<10} {'Status':<12} {'Reason':<25} {'Threshold':<12} {'Val OK FPR %':<15}")
print("-"*100)
for result in training_results:
    conn_id = result['connector_id']
    n_imgs = result['n_images']
    status = result['status']
    reason = result.get('reason', '') or ''
    threshold_str = f"{result['threshold']:.4f}" if result['threshold'] else "N/A"
    fpr_str = f"{result['val_ok_fpr_estimate']:.2f}%" if result.get('val_ok_fpr_estimate') is not None else "N/A"
    
    # Truncate long reasons
    if len(reason) > 24:
        reason = reason[:21] + "..."
    
    print(f"{conn_id:<12} {n_imgs:<10} {status:<12} {reason:<25} {threshold_str:<12} {fpr_str:<15}")
print("="*100)

# Statistics
trained_count = sum(1 for r in training_results if r['status'] == 'TRAINED')
skipped_count = sum(1 for r in training_results if r['status'] == 'SKIPPED')
error_count = sum(1 for r in training_results if r['status'] == 'ERROR')
total_images = sum(r['n_images'] for r in training_results)

print(f"\n📊 Statistics:")
print(f"  Total connectors processed: {len(training_results)}")
print(f"  ✅ Successfully trained: {trained_count}")
print(f"  ⏭️  Skipped: {skipped_count}")
print(f"  ❌ Errors: {error_count}")
print(f"  📷 Total images: {total_images}")
print("="*100)


## 12. Test Image Classification

Classify images from the "Test" folder using the trained models.


In [ ]:
def load_trained_model(connector_id: str, output_root: Path, device: torch.device) -> Tuple[nn.Module, float]:
    """Load trained model and threshold for a connector."""
    model_dir = output_root / connector_id
    
    # Load threshold
    threshold_path = model_dir / "threshold.json"
    if not threshold_path.exists():
        raise FileNotFoundError(f"Threshold file not found for {connector_id}")
    
    with open(threshold_path, 'r') as f:
        threshold_data = json.load(f)
    threshold = threshold_data['threshold']
    
    # Load model
    model_path = model_dir / "model.pt"
    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found for {connector_id}")
    
    model = create_model(input_channels=1, pretrained=False)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    
    return model, threshold

def classify_test_images(
    test_folder: Path,
    connector_masks: Dict,
    output_root: Path,
    device: torch.device,
    connector_id: str = "conn1"
):
    """
    Classify images from test folder using trained models.
    
    Assumes test images are PNG files directly in the Test folder.
    Default connector is conn1, but can be specified.
    """
    results = []
    
    # Get PNG images directly from Test folder
    img_files = sorted(list(test_folder.glob("*.png")))
    
    if not img_files:
        print(f"No PNG images found in {test_folder}")
        return results
    
    print(f"Found {len(img_files)} PNG images in Test folder")
    
    # Check if connector is available
    if connector_id not in connector_masks:
        print(f"Error: {connector_id} not found in trained models")
        print(f"Available connectors: {list(connector_masks.keys())}")
        return results
    
    # Load model and threshold
    try:
        model, threshold = load_trained_model(connector_id, output_root, device)
        print(f"\nLoaded model for {connector_id}, threshold: {threshold:.4f}")
    except Exception as e:
        print(f"Error loading model for {connector_id}: {e}")
        return results
    
    # Classify images
    print(f"\nClassifying {len(img_files)} images as {connector_id}...")
    print("="*80)
    
    for idx, img_path in enumerate(img_files):
        try:
            prob_ko, is_ko = predict_connector(
                img_path, connector_id, model, connector_masks[connector_id], threshold, device
            )
            prediction = 'KO' if is_ko else 'OK'
            
            results.append({
                'connector': connector_id,
                'image': img_path.name,
                'path': str(img_path),
                'prob_ko': prob_ko,
                'is_ko': is_ko,
                'prediction': prediction
            })
            
            # Visualize result
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if img is None:
                img = np.array(Image.open(img_path).convert('L'))
            
            # Create visualization
            fig, ax = plt.subplots(1, 1, figsize=(8, 6))
            ax.imshow(img, cmap='gray')
            
            # Color based on prediction
            color = 'red' if is_ko else 'green'
            title = f"{img_path.name}\nPrediction: {prediction} | Prob KO: {prob_ko:.4f} | Threshold: {threshold:.4f}"
            ax.set_title(title, fontsize=12, color=color, weight='bold')
            ax.axis('off')
            
            # Add border based on prediction
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(3)
            
            plt.tight_layout()
            plt.show()
            
            print(f"\n[{idx+1}/{len(img_files)}] {img_path.name}")
            print(f"  Probabilità KO: {prob_ko:.4f}")
            print(f"  Soglia: {threshold:.4f}")
            print(f"  Predizione: {prediction}")
            print("-"*80)
            
        except Exception as e:
            print(f"\nError processing {img_path.name}: {e}")
            import traceback
            traceback.print_exc()
            results.append({
                'connector': connector_id,
                'image': img_path.name,
                'path': str(img_path),
                'prob_ko': None,
                'is_ko': None,
                'prediction': f'ERROR: {str(e)}'
            })
            print("-"*80)
    
    return results

# Test folder path (in Colab current directory)
TEST_FOLDER = Path("./test2")

if not TEST_FOLDER.exists():
    print(f"Warning: Test folder not found at {TEST_FOLDER}")
    print("Please ensure the test2 folder exists in the current Colab directory")
    print("Expected structure: test2/ contains PNG images of conn2")
else:
    print(f"Classifying images from: {TEST_FOLDER}")
    print("="*80)
    
    # Classify images as conn2
    test_results = classify_test_images(
        TEST_FOLDER,
        connector_masks,
        OUTPUT_ROOT,
        device,
        connector_id="conn2"
    )
    
    # Print detailed summary
    print("\n" + "="*80)
    print("CLASSIFICATION SUMMARY")
    print("="*80)
    print(f"{'Connector':<12} {'Image':<40} {'Prob KO':<12} {'Threshold':<12} {'Prediction':<10}")
    print("-"*80)
    
    ok_count = 0
    ko_count = 0
    error_count = 0
    
    # Load threshold for display
    try:
        threshold_path = OUTPUT_ROOT / connector_id / "threshold.json"
        with open(threshold_path, 'r') as f:
            threshold_data = json.load(f)
        threshold_val = threshold_data['threshold']
    except:
        threshold_val = None
    
    for result in test_results:
        prob_str = f"{result['prob_ko']:.4f}" if result['prob_ko'] is not None else "N/A"
        threshold_str = f"{threshold_val:.4f}" if threshold_val is not None else "N/A"
        pred = result['prediction']
        
        if 'ERROR' in pred:
            error_count += 1
        elif pred == 'KO':
            ko_count += 1
        else:
            ok_count += 1
        
        # Color coding in print (if terminal supports it)
        pred_display = pred
        if pred == 'KO':
            pred_display = f"❌ {pred}"
        elif pred == 'OK':
            pred_display = f"✅ {pred}"
        
        print(f"{result['connector']:<12} {result['image']:<40} {prob_str:<12} {threshold_str:<12} {pred_display:<15}")
    
    print("-"*80)
    print(f"\n📊 STATISTICS:")
    print(f"  Total images processed: {len(test_results)}")
    print(f"  ✅ OK: {ok_count}")
    print(f"  ❌ KO: {ko_count}")
    print(f"  ⚠️  Errors: {error_count}")
    print("="*80)
    
    # Save results to CSV
    if test_results:
        import pandas as pd
        df_results = pd.DataFrame(test_results)
        results_csv_path = OUTPUT_ROOT / "test_classification_results.csv"
        df_results.to_csv(results_csv_path, index=False)
        print(f"\nResults saved to: {results_csv_path}")
